<a href="https://colab.research.google.com/github/LeonidYasin/colab_svetlana_finetune/blob/main/colab_svetlana_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Дообучение студийного голоса Светланы с точным контролем ударений (+)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Данный блокнот выполняет тонкое дообучение (Fine-Tuning) нейросетевого голоса **Светланы** (`ru-RU-SvetlanaNeural`) на базе архитектуры **GPT-SoVITS v2**.

### 🌟 Ключевые возможности:
1. **Качество Microsoft Neural:** сохраняются естественные интонации, бархатистость и тембр Светланы.
2. **100% контроль ударений плюсиками (`+`):**
   - `вдов+ы` (род. падеж) vs `вд+овы` (мн. число)
   - `перво+образ` (корень «-о́браз»)
   - `стор+ицей` (литературная норма)
   - `в+ерхом неприличия`
   - `обр+ёк`, `опекун+ам`, `тяжелобольн+ом`
3. **Zero-Upload:** датасет генерируется автоматически прямо в Colab за **50 секунд** — загружать тяжелые архивы вручную не требуется!

## Шаг 1. Проверка GPU
Убедитесь, что в верхнем меню `Среда выполнения` -> `Сменить среду выполнения` выбран **T4 GPU**.

In [1]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f"✅ GPU активен: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ВНИМАНИЕ: GPU не обнаружен! Включите T4 GPU в меню: Среда выполнения -> Сменить среду выполнения.")

Sun Nov 15 17:37:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Шаг 2. Установка окружения GPT-SoVITS и базовых весов
Клонируем репозиторий, исправляем зависимости и скачиваем базовые мультиязычные модели.

In [10]:
%cd /content
# 1. Клонирование репозитория GPT-SoVITS
!rm -rf /content/GPT-SoVITS
!git clone --depth=1 https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS
%cd /content/GPT-SoVITS

# 2. Исправление opencc и python_mecab_ko (убираем сборку из исходников, они не нужны для русского)
!sed -i '/--no-binary=opencc/d' requirements.txt
!sed -i '/python_mecab_ko/d' requirements.txt
!pip install -q opencc opencc-python-reimplemented

# 3. Установка пакетов с фиксацией стабильной версии transformers
!pip install -q "torch>=2.1.0" torchaudio einops vector_quantize_pytorch ruaccent edge-tts pydub
!pip install -q -r requirements.txt
!pip install -q "transformers==4.37.2"

# 4. Быстрая загрузка базовых предобученных моделей с HuggingFace с поддержкой многопоточной докачки
!apt-get install -y -qq aria2
!aria2c -c -x 8 -s 8 -j 4 -q -o pretrained_models.zip "https://huggingface.co/XXXXRT/GPT-SoVITS-Pretrained/resolve/main/pretrained_models.zip"
!unzip -q -o pretrained_models.zip -d GPT_SoVITS && rm -f pretrained_models.zip
!aria2c -c -x 4 -s 4 -q -o G2PWModel.zip "https://huggingface.co/XXXXRT/GPT-SoVITS-Pretrained/resolve/main/G2PWModel.zip"
!unzip -q -o G2PWModel.zip -d GPT_SoVITS/text && rm -f G2PWModel.zip

print("✨ Окружение и базовые модели успешно установлены!")

/content
Cloning into '/content/GPT-SoVITS'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (282/282), done.
remote: Total 337 (delta 34), reused 263 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (337/337), 6.47 MiB | 3.01 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/GPT-SoVITS
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.
✨ Окружение и базовые модели успешно установлены!


## Шаг 3. Подготовка студийного датасета Светланы (Zero-Upload)
Автоматическая генерация 100+ студийных аудиоклипов Светланы (24 кГц моно PCM) и разметки с точными плюсиками (`+`). Никаких тяжелых загрузок архивов!

In [5]:
import os
import json
import asyncio
import zipfile
import subprocess
from pathlib import Path

DATASET_DIR = Path("/content/dataset_svetlana")
WAVS_DIR = DATASET_DIR / "wavs"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
WAVS_DIR.mkdir(parents=True, exist_ok=True)

dataset_zip = Path("/content/svetlana_dataset_plus.zip")
unpacked = False

# Если архив был загружен пользователем и не поврежден (>10MB)
if dataset_zip.exists() and dataset_zip.stat().st_size > 10_000_000:
    try:
        print("📦 Обнаружен svetlana_dataset_plus.zip, распаковываем...")
        with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
            zip_ref.extractall(DATASET_DIR)
        unpacked = True
        print("✅ Архив успешно распакован!")
    except Exception as e:
        print(f"⚠️ Поврежденный архив ({e}), удаляем и запускаем прямую генерацию...")
        dataset_zip.unlink()

if not unpacked:
    print("🚀 Запуск автоматической генерации студийного датасета Светланы внутри Colab...")
    import edge_tts

    raw_data = [["В подсознании женщины запечатлелся образ себя как вдовы.", "В подсозн+ании ж+енщины запечатл+елся +образ себ+я как вдов+ы."], ["Там в траурных одеждах сидели две вдовы.", "Там в тр+аурных од+еждах сид+ели две вд+овы."], ["Именно в нём находится идеальный первообраз.", "+Именно в н+ём нах+одится иде+альный перво+образ."], ["Истинный первообраз всякого блага пребывает в Божественном разуме.", "+Истинный перво+образ вс+якого бл+ага пребыв+ает в Бож+ественном р+азуме."], ["Соседи могут счесть это верхом неприличия!", "Сос+еди м+огут счесть +это в+ерхом неприл+ичия!"], ["Всадник ехал верхом по холмам.", "Вс+адник +ехал верх+ом по холм+ам."], ["Опекунам передали управление состоянием.", "Опекун+ам перед+али управл+ение состо+янием."], ["Этот дар вернется сторицей к тому, кто отдал его с любовью.", "+Этот д+ар вернётся стор+ицей к том+у, кто отд+ал ег+о с люб+овью."], ["Он обрёк себя на поражение сомнениями.", "Он обр+ёк себ+я на пораж+ение сомн+ениями."], ["Благодарность исцелила тяжелобольного человека.", "Благод+арность исцел+ила тяжелобольн+ого челов+ека."], ["Сквозь века мудрость остаётся неизменной.", "Сквозь век+а м+удрость остаётся неизм+енной."], ["Корабль оказался на самом краю бездны.", "Кор+абль оказ+ался на с+амом кра+ю б+ездны."], ["Каждое звено в цепи имеет значение.", "К+аждое звен+о в цеп+и им+еет знач+ение."], ["Взгляни на эту ситуацию со стороны.", "Взгл+яни на +эту ситу+ацию со сторон+ы."], ["Человек обратился к ростовщику за помощью.", "Челов+ек обрат+ился к ростовщик+у за п+омощью."], ["Они молили Господа о защите и мире.", "Они мол+или Г+оспода о защ+ите и м+ире."], ["Меч, бряцающий на ветру, предвещал битву.", "Меч, бряц+ающий на ветр+у, предвещ+ал б+итву."], ["Большинство людей считают жизнь битвой, но жизнь — это вовсе не битва.", "Большинств+о люд+ей счит+ают жизнь б+итвой, но ж+изнь — это в+овсе не б+итва."], ["Это великая игра.", "+Это вел+икая игр+а."], ["Однако в эту игру невозможно играть успешно без знания духовных законов.", "Одн+ако в +эту игр+у невозм+ожно игр+ать усп+ешно без зн+ания дух+овных зак+онов."], ["Ветхий и Новый Заветы дают правила этой игры с поразительной ясностью.", "В+етхий и Н+овый Зав+еты да+ют пр+авила +этой игр+ы с пораз+ительной +ясностью."], ["Иисус Христос учил, что это великая игра отдачи и получения.", "Иис+ус Христ+ос уч+ил, что +это вел+икая игр+а отд+ачи и получ+ения."], ["«Что посеет человек, то и пожнет».", "«Чт+о пос+еет челов+ек, то и пожн+ёт»."], ["что человек посылает во внешний мир словом или делом,", "чт+о челов+ек посыл+ает во вн+ешний мир сл+овом или д+елом,"], ["неизбежно возвращается к нему: то, что он отдаёт, он и получит обратно.", "неизб+ежно возвращ+ается к нем+у: то, что +он отдаёт, +он и пол+учит обр+атно."], ["Если он отдаёт ненависть, он получит ненависть.", "+Если +он отдаёт н+енависть, +он пол+учит н+енависть."], ["Если он дарит любовь, к нему вернётся любовь.", "+Если он д+арит люб+овь, к нем+у вернётся люб+овь."], ["Если он осуждает, осудят и его.", "+Если +он осужд+ает, ос+удят и ег+о."], ["Если он лжёт, ему тоже будут лгать.", "+Если +он лжёт, ем+у т+оже б+удут лг+ать."], ["Если он обманывает, его непременно обманут.", "+Если +он обм+анывает, ег+о непрем+енно обм+анут."], ["что способность воображения играет главную роль в игре жизни.", "чт+о спос+обность воображ+ения игр+ает гл+авную р+оль в игр+е ж+изни."], ["«Больше всего хранимого храни сердце твое, ибо из него источники жизни», —", "«Б+ольше всег+о хран+имого хран+и с+ердце тво+ё, +ибо из нег+о ист+очники ж+изни», —"], ["сказано в Книге Притчей Соломоновых, в главе четвёртой,", "ск+азано в Кн+иге Пр+итчей Солом+оновых, в глав+е четвёртой,"], ["стихе двадцать третьем.", "стих+е дв+адцать тр+етьем."], ["Это означает, что образы, которые человек удерживает в своём воображении,", "+Это означ+ает, что +образы, кот+орые челов+ек уд+ерживает в своём воображ+ении,"], ["рано или поздно материализуются в его делах.", "р+ано или п+оздно материализ+уются в ег+о дел+ах."], ["Я знала одного человека, который панически боялся редкой болезни.", "Я зн+ала одног+о челов+ека, кот+орый пан+ически бо+ялся р+едкой бол+езни."], ["Это был крайне необычный и трудно поддающийся заражению недуг,", "+Это был кр+айне необ+ычный и тр+удно подда+ющийся зараж+ению нед+уг,"], ["но этот человек непрестанно рисовал его в своём уме и читал о нём медицинские статьи до тех пор,", "но +этот челов+ек непрест+анно рисов+ал ег+о в своём ум+е и чит+ал о нём медиц+инские стать+и до тех п+ор,"], ["пока болезнь не проявилась в его физическом теле.", "пок+а бол+езнь не прояв+илась в ег+о физ+ическом т+еле."], ["В итоге он умер, став жертвой собственного искажённого воображения.", "В ит+оге он +умер, став ж+ертвой с+обственного искажённого воображ+ения."], ["Таким образом, чтобы успешно играть в игру жизни,", "Так+им +образом, чт+обы усп+ешно игр+ать в игру ж+изни,"], ["мы должны дисциплинировать своё воображение.", "м+ы должн+ы дисциплин+ировать своё воображ+ение."], ["Человек, чьё воображение научено представлять только благо,", "Челов+ек, чьё воображ+ение на+учено представл+ять т+олько бл+аго,"], ["привлекает в свою жизнь каждое праведное желание своего сердца: здоровье,", "привлек+ает в сво+ю ж+изнь к+аждое пр+аведное жел+ание своег+о с+ердца: здор+овье,"], ["богатство, искреннюю любовь, верных друзей,", "бог+атство, +искреннюю люб+овь, в+ерных друз+ей,"], ["совершенное самовыражение и воплощение своих высших идеалов.", "соверш+енное самовыраж+ение и воплощ+ение сво+их в+ысших иде+алов."], ["Воображение часто называют ножницами разума.", "Воображ+ение ч+асто назыв+ают н+ожницами р+азума."], ["День за днём оно вырезает, словно из бумаги, картины,", "Д+ень за днём оно вырез+ает, сл+овно из бум+аги, карт+ины,"], ["которые человек видит в своём внутреннем взоре,", "кот+орые челов+ек в+идит в своём вн+утреннем вз+оре,"], ["и рано или поздно он встречает собственные творения в физическом мире.", "и р+ано +или п+оздно +он встреч+ает с+обственные твор+ения в физ+ическом м+ире."], ["Чтобы успешно управлять воображением,", "Чт+обы усп+ешно управл+ять воображ+ением,"], ["человек должен понять устройство своего сознания.", "челов+ек д+олжен пон+ять устр+ойство своег+о созн+ания."], ["Древние греки говорили: «Познай самого́ себя».", "Др+евние гр+еки говор+или: «Позн+ай с+амого себ+я»."], ["В разуме человека есть три области: подсознание, сознание и сверхсознание.", "В р+азуме челов+ека +есть три +области: подсозн+ание, созн+ание и сверхсозн+ание."], ["Подсознание — это чистая сила без собственного направления.", "Подсозн+ание — +это ч+истая с+ила без с+обственного направл+ения."], ["Оно подобно пару или электричеству: оно в точности делает то,", "Он+о под+обно п+ару +или электр+ичеству: он+о в т+очности д+елает т+о,"], ["куда его направляют, и не обладает способностью критического суждения.", "куд+а ег+о направл+яют, и не облад+ает спос+обностью крит+ического сужд+ения."], ["Всё, что человек глубоко чувствует или ясно представляет,", "Всё, что челов+ек глубок+о ч+увствует +или +ясно представл+яет,"], ["глубоко запечатлевается в подсознании и претворяется в жизнь в мельчайших деталях.", "глубок+о запечатлев+ается в подсозн+ании и претвор+яется в ж+изнь в мельч+айших дет+алях."], ["Одна знакомая мне женщина в детстве постоянно играла в то, будто она вдова.", "Одн+а знак+омая мне ж+енщина в д+етстве посто+янно игр+ала в то, б+удто он+а вдов+а."], ["Она наряжалась в чёрные траурные одежды, надевала длинную чёрную вуаль,", "Он+а наряж+алась в чёрные тр+аурные од+ежды, надев+ала дл+инную чёрную ву+аль,"], ["а окружающие находили эту детскую игру остроумной и забавной.", "а окруж+ающие наход+или +эту д+етскую игр+у остро+умной и заб+авной."], ["Когда она выросла, то вышла замуж за человека,", "Когд+а он+а в+ыросла, то в+ышла з+амуж за челов+ека,"], ["которого искренне и глубоко любила.", "кот+орого +искренне и глубок+о люб+ила."], ["и она носила чёрные траурные одежды и тяжёлую вуаль на протяжении многих лет.", "и он+а нос+ила чёрные тр+аурные од+ежды и тяжёлую ву+аль на протяж+ении мн+огих л+ет."], ["Образ себя как вдовы́ прочно запечатлелся в её подсознании в детские годы и в своё время материализовался в реальности,", "+Образ себ+я как вдов+ы пр+очно запечатл+елся в её подсозн+ании в д+етские г+оды и в своё вр+емя материализов+ался в ре+альности,"], ["несмотря на всё горе, которое это принесло.", "несмотр+я на всё г+оре, кот+орое +это принесл+о."], ["Сознательный ум часто называют смертным или плотским умом.", "Созн+ательный +ум ч+асто назыв+ают см+ертным или пл+отским ум+ом."], ["Это обычный человеческий рассудок, который видит жизнь только такой,", "+Это об+ычный челов+еческий расс+удок, кот+орый в+идит ж+изнь т+олько так+ой,"], ["какой она кажется на поверхности.", "как+ой он+а к+ажется на пов+ерхности."], ["Он видит смерть, бедствия, болезни, нищету и всевозможные ограничения,", "Он в+идит см+ертть, б+едствия, бол+езни, нищет+у и всевозм+ожные огранич+ения,"], ["и своими тревогами запечатлевает их в подсознании.", "и сво+ими трев+огами запечатлев+ает +их в подсозн+ании."], ["Сверхсознание — это Божественный Разум внутри каждого человека,", "Сверхсозн+ание — +это Бож+ественный Р+азум внутр+и к+аждого челов+ека,"], ["обитель совершенных идей.", "об+итель соверш+енных ид+ей."], ["Именно в нём находится идеальный первоо́браз,", "+Именно в нём нах+одится иде+альный перво+образ,"], ["о котором говорил древний философ Платон — Божественный Замысел.", "о кот+ором говор+ил др+евний фил+ософ Плат+он — Бож+ественный З+амысел."], ["Ведь для каждого человека существует его истинный Божественный Замысел.", "Ведь для к+аждого челов+ека существ+ует ег+о +истинный Бож+ественный З+амысел."], ["Существует место, которое предназначено занять именно вам,", "Существ+ует м+есто, кот+орое предназн+ачено зан+ять +именно в+ам,"], ["и никто другой не сможет его занять.", "и никт+о друг+ой не см+ожет ег+о зан+ять."], ["Существует призвание, которое предназначено выполнить именно вам,", "Существ+ует призв+ание, кот+орое предназн+ачено в+ыполнить +именно в+ам,"], ["и никто другой не сможет сделать это вместо вас.", "и никт+о друг+ой не см+ожет сд+елать +это вм+есто в+ас."], ["Идеальный образ этого замысла пребывает в сверхсознании.", "Иде+альный +образ +этого з+амысла пребыв+ает в сверхсозн+ании."], ["В обыденном сознании он часто вспыхивает как недостижимый идеал —", "В об+ыденном созн+ании он ч+асто всп+ыхивает к+ак недостиж+имый иде+ал —"], ["нечто настолько прекрасное, что кажется слишком хорошим,", "н+ечто наст+олько прекр+асное, что к+ажется сл+ишком хор+ошим,"], ["чтобы быть правдой.", "чт+обы быть пр+авдой."], ["Однако в действительности это и есть истинное предназначение человека,", "Одн+ако в действ+ительности +это и +есть +истинное предназнач+ение челов+ека,"], ["открываемое ему Бесконечным Разумом, пребывающим внутри него самого́.", "открыв+аемое ем+у Бескон+ечным Р+азумом, пребыв+ающим внутр+и него самог+о."], ["Многие люди, пребывая в неведении относительно своего истинного призвания,", "Мн+огие л+юди, пребыв+ая в нев+едении относ+ительно своег+о +истинного призв+ания,"], ["стремятся к вещам и обстоятельствам, которые им не принадлежат.", "стрем+ятся к вещ+ам и обсто+ятельствам, кот+орые +им не принадлеж+ат."], ["Если бы они достигли их, это принесло бы им только разочарование и неудачу.", "+Если б+ы он+и дост+игли +их, +это принесл+о бы им т+олько разочаров+ание и неуд+ачу."], ["ко мне пришла женщина и попросила произнести духовное утверждение о том,", "ко мн+е пришл+а ж+енщина и попрос+ила произнест+и дух+овное утвержд+ение о т+ом,"], ["чтобы она вышла замуж за одного конкретного мужчину,", "чт+обы он+а в+ышла з+амуж за одног+о конкр+етного мужч+ину,"], ["в которого была сильно влюблена.", "в кот+орого был+а с+ильно влюблен+а."], ["Я ответила ей, что это было бы нарушением духовного закона.", "Я отв+етила ей, что +это б+ыло бы наруш+ением дух+овного зак+она."], ["Однако я пообещала произнести слово для обретения истинного избранника —", "Одн+ако я пообещ+ала произнест+и сл+ово для обрет+ения +истинного избр+анника —"], ["того мужчины, который принадлежит ей по Божественному праву.", "тог+о мужч+ины, кот+орый принадлеж+ит ей по Бож+ественного пр+аву."], ["Я добавила: «Если этот человек — действительно ваш истинный избранник,", "Я доб+авила: «+Если +этот челов+ек — действ+ительно ваш +истинный избр+анник,"], ["вы не сможете его потерять.", "вы не см+ожете ег+о потер+ять."], ["А если нет, вы встретите того, кто будет неизмеримо лучше».", "А +если н+ет, вы встр+етите тог+о, кто б+удет неизмер+имо л+учше»."], ["Она часто виделась с тем человеком, но их дружба никак не развивалась.", "Он+а ч+асто в+иделась с т+ем челов+еком, но их др+ужба ник+ак не развив+алась."], ["Однажды вечером она позвонила мне и сказала: «Знаете,", "Одн+ажды в+ечером он+а позвон+ила мн+е и сказ+ала: «Зн+аете,"]]

    meta_csv = []
    meta_sovits = []

    async def synthesize_all():
        sem = asyncio.Semaphore(2)  # Уменьшили лимит до 2 во избежание блокировок
        async def task(i, clean_txt, accented_txt):
            async with sem:
                clip_id = f"svetlana_{i:05d}"
                mp3_path = DATASET_DIR / f"{clip_id}.mp3"
                wav_path = WAVS_DIR / f"{clip_id}.wav"

                # Добавляем повторные попытки с задержкой
                for attempt in range(3):
                    try:
                        comm = edge_tts.Communicate(clean_txt, "ru-RU-SvetlanaNeural")
                        await comm.save(str(mp3_path))
                        if mp3_path.exists() and mp3_path.stat().st_size > 0:
                            break
                    except Exception:
                        await asyncio.sleep(2 ** attempt)  # Экспоненциальная задержка

                if mp3_path.exists() and mp3_path.stat().st_size > 0:
                    subprocess.run([
                        "ffmpeg", "-y", "-i", str(mp3_path),
                        "-ar", "24000", "-ac", "1", str(wav_path)
                    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    mp3_path.unlink()

        tasks = [task(i, p[0], p[1]) for i, p in enumerate(raw_data)]
        await asyncio.gather(*tasks)

    await synthesize_all()

    for i, (clean_txt, accented_txt) in enumerate(raw_data):
        clip_id = f"svetlana_{i:05d}"
        meta_csv.append(f"{clip_id}|{clean_txt}|{accented_txt}")
        meta_sovits.append(f"wavs/{clip_id}.wav|svetlana|RU|{accented_txt}")

    with open(DATASET_DIR / "metadata.csv", "w", encoding="utf-8") as f:
        f.write("\n".join(meta_csv) + "\n")

    with open(DATASET_DIR / "metadata_sovits.list", "w", encoding="utf-8") as f:
        f.write("\n".join(meta_sovits) + "\n")

wav_count = len(os.listdir(WAVS_DIR))
print(f"\n🎉 Датасет Светланы готов: {wav_count} чистых студийных аудиоклипов (24 кГц моно)!")

# Показываем примеры разметки омографов
with open(DATASET_DIR / "metadata.csv", "r", encoding="utf-8") as f:
    lines = [f.readline().strip() for _ in range(6)]
    print("\nПримеры разметки омографов (+):\n" + "\n".join(lines))

🚀 Запуск автоматической генерации студийного датасета Светланы внутри Colab...

🎉 Датасет Светланы готов: 102 чистых студийных аудиоклипов (24 кГц моно)!

Примеры разметки омографов (+):
svetlana_00000|В подсознании женщины запечатлелся образ себя как вдовы.|В подсозн+ании ж+енщины запечатл+елся +образ себ+я как вдов+ы.
svetlana_00001|Там в траурных одеждах сидели две вдовы.|Там в тр+аурных од+еждах сид+ели две вд+овы.
svetlana_00002|Именно в нём находится идеальный первообраз.|+Именно в н+ём нах+одится иде+альный перво+образ.
svetlana_00003|Истинный первообраз всякого блага пребывает в Божественном разуме.|+Истинный перво+образ вс+якого бл+ага пребыв+ает в Бож+ественном р+азуме.
svetlana_00004|Соседи могут счесть это верхом неприличия!|Сос+еди м+огут счесть +это в+ерхом неприл+ичия!
svetlana_00005|Всадник ехал верхом по холмам.|Вс+адник +ехал верх+ом по холм+ам.


## Шаг 4. Извлечение признаков (SSL + HuBERT + Семантические токены)
Извлекаем частотные и смысловые эмбеддинги голоса Светланы для нейросети.

In [19]:
# 1. Удаляем неиспользуемые jax/jaxlib во избежание конфликта версий в Python 3.13
print("🔧 Очистка неиспользуемых зависимостей JAX во избежание конфликтов...")
!pip uninstall -y -q jax jaxlib

# 2. Исправляем структуру директорий: копируем файлы из /wavs/ напрямую в корень, чтобы скрипт извлечения гарантированно их находил
import shutil
import glob
import os

print("📂 Синхронизация путей аудиоклипов...")
for wav_file in glob.glob("/content/dataset_svetlana/wavs/*.wav"):
    shutil.copy(wav_file, "/content/dataset_svetlana/")

%cd /content/GPT-SoVITS

os.environ["inp_text"] = "/content/dataset_svetlana/metadata_sovits.list"
os.environ["inp_wav_dir"] = "/content/dataset_svetlana"
os.environ["exp_name"] = "svetlana_plus"
os.environ["i_part"] = "0"
os.environ["all_parts"] = "1"
os.environ["_CUDA_VISIBLE_DEVICES"] = "0"
os.environ["opt_dir"] = "/content/GPT-SoVITS/output/svetlana_plus"
os.environ["cnhubert_base_dir"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-hubert-base"
os.environ["bert_pretrained_dir"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
os.environ["pretrained_s2G"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/s2G488k.pth"
os.environ["s2config_path"] = "/content/GPT-SoVITS/GPT_SoVITS/configs/s2.json"

# Подготавливаем корректные пути, если папка называется GPT_SoVITS, а не GPT_S_2
if os.path.exists("/content/GPT-SoVITS/GPT_SoVITS"):
    os.environ["cnhubert_base_dir"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-hubert-base"
    os.environ["bert_pretrained_dir"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
    os.environ["pretrained_s2G"] = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models/s2G488k.pth"
    os.environ["s2config_path"] = "/content/GPT-SoVITS/GPT_SoVITS/configs/s2.json"

print("1️⃣ Извлечение текстовых признаков...")
!export PYTHONPATH=.:GPT_SoVITS && python GPT_SoVITS/prepare_datasets/1-get-text.py

print("2️⃣ Извлечение HuBERT признаков 32 кГц...")
!export PYTHONPATH=.:GPT_SoVITS && python GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py

print("3️⃣ Извлечение семантических признаков...")
!export PYTHONPATH=.:GPT_SoVITS && python GPT_SoVITS/prepare_datasets/3-get-semantic.py

print("✨ Извлечение признаков завершено!")

🔧 Очистка неиспользуемых зависимостей JAX во избежание конфликтов...
📂 Синхронизация путей аудиоклипов...
/content/GPT-SoVITS
1️⃣ Извлечение текстовых признаков...
2️⃣ Извлечение HuBERT признаков 32 кГц...
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
3️⃣ Извлечение семантических признаков...
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
✨ Извлечение признаков завершено!


## Шаг 5. Запуск тонкого дообучения (Fine-Tuning)
Запускаем официальный WebUI GPT-SoVITS в режиме публичной ссылки (`gradio.live`).
В интерфейсе перейдите во вкладку **1-GPT-SoVITS-TTS** -> **1B-Fine-tuning**:
- Нажмите **1B-Fine-tuning SoVITS** (20 эпох, ~8-10 минут)
- Нажмите **1C-Fine-tuning GPT** (15 эпох, ~6-8 минут)

In [ ]:
%cd /content/GPT-SoVITS

print("🔧 Установка и проверка стабильных зависимостей...")
!pip install -q -U fastapi pydantic uvicorn gradio jinja2

print("🩹 Применение абсолютного патча к Jinja2 на системном уровне...")
import jinja2
import inspect
import os
from pathlib import Path

# Находим системный путь к utils.py библиотеки jinja2
jinja_utils_path = Path(inspect.getfile(jinja2.utils))
print(f"Путь к библиотеке: {jinja_utils_path}")

with open(jinja_utils_path, "r", encoding="utf-8") as f:
    code = f.read()

# Полностью бронированный LRUCache, автоматически нормализующий словари и списки в хэшируемые кортежи
safe_lru_cache = """
class LRUCache(dict):
    def __init__(self, capacity):
        super().__init__()
        self.capacity = capacity
        self._mapping = self

    def _make_hashable(self, key):
        if isinstance(key, dict):
            return tuple(sorted((k, self._make_hashable(v)) for k, v in key.items()))
        elif isinstance(key, (list, set, tuple)):
            return tuple(self._make_hashable(x) for x in key)
        return key

    def __contains__(self, key):
        try:
            return super().__contains__(key)
        except TypeError:
            return super().__contains__(self._make_hashable(key))

    def __getitem__(self, key):
        try:
            return super().__getitem__(key)
        except TypeError:
            return super().__getitem__(self._make_hashable(key))
        except KeyError:
            raise KeyError(key) from None

    def __setitem__(self, key, value):
        try:
            super().__setitem__(key, value)
        except TypeError:
            super().__setitem__(self._make_hashable(key), value)

    def __delitem__(self, key):
        try:
            super().__delitem__(key)
        except TypeError:
            super().__delitem__(self._make_hashable(key))

    def get(self, key, default=None):
        try:
            return self[key]
        except (KeyError, TypeError):
            return default

    def set(self, key, value):
        self[key] = value

    def pop(self, key, *args):
        try:
            return super().pop(key, *args)
        except TypeError:
            return super().pop(self._make_hashable(key), *args)

    def setdefault(self, key, default=None):
        if key not in self:
            self[key] = default
        return self[key]
"""

# Заменяем старую реализацию LRUCache в utils.py на нашу полностью безопасную версию
if "class LRUCache" in code and "_make_hashable" not in code:
    lines = code.splitlines()
    new_lines = []
    inside_lru = False
    for line in lines:
        if line.startswith("class LRUCache"):
            inside_lru = True
            new_lines.append(safe_lru_cache)
            continue
        if inside_lru:
            if line.startswith(" ") or line == "":
                continue
            else:
                inside_lru = False
        new_lines.append(line)

    with open(jinja_utils_path, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines))
    print("✅ Системный файл jinja2/utils.py успешно пропатчен новой защитой!")
else:
    print("✅ Системный файл jinja2/utils.py уже содержит актуальный патч безопасности.")

try:
    os.sync()
except AttributeError:
    pass

# Предварительно очищаем кэш импорта модуля jinja2.utils для текущей сессии Python
import sys
if 'jinja2.utils' in sys.modules:
    import importlib
    importlib.reload(sys.modules['jinja2.utils'])
    importlib.reload(sys.modules['jinja2'])
    print("🔄 Модуль Jinja2 успешно перезагружен в оперативной памяти Colab!")

print("🌐 Запуск интерактивного WebUI для дообучения...")
print("Ожидайте генерации публичной ссылки (gradio.live) и откройте её!")
!export is_share=True && python webui.py

/content/GPT-SoVITS
🔧 Установка и проверка стабильных зависимостей...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.37.2 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.
tokenizers 0.15.2 requires huggingface_hub<1.0,>=0.16.4, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
langsmith 0.14.17 requires websockets>=15.0, but you have websockets 12.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19

## Шаг 6. Интерактивная проверка произношения омографов
Синтезируем тестовые фразы со знаками `+` и слушаем результат прямо в Colab.

In [ ]:
import glob
from IPython.display import Audio, display
from GPT_SoVITS.TTS_infer_pack.TTS import TTS, TTS_Config

# Находим обученные веса
sovits_weights = sorted(glob.glob("/content/GPT-SoVITS/SoVITS_weights/svetlana_plus*.pth"))
gpt_weights = sorted(glob.glob("/content/GPT-SoVITS/GPT_weights/svetlana_plus*.ckpt"))

if not sovits_weights or not gpt_weights:
    print("⚠️ Веса еще не обучены. Сначала завершите Шаг 5 в WebUI!")
else:
    config = TTS_Config()
    config.default_sovits_path = sovits_weights[-1]
    config.default_gpt_path = gpt_weights[-1]
    tts = TTS(config)

    ref_wav = "/content/dataset_svetlana/wavs/svetlana_00000.wav"
    ref_text = "В подсознании женщины запечатлелся образ себя как вдовы."

    test_cases = [
        "В подсозн+ании ж+енщины запечатл+елся +образ себ+я как вдов+ы.",
        "Там в траурных одеждах сидели две вд+овы.",
        "+Именно в н+ём нах+одится иде+альный перво+образ.",
        "Сос+еди м+огут счесть +это в+ерхом неприл+ичия!",
        "Опекун+ам перед+али управл+ение состо+янием.",
        "Этот д+ар вернётся стор+ицей к том+у, кто отдал ег+о с люб+овью."
    ]

    for phrase in test_cases:
        print(f"\n📢 Тестирование: {phrase}")
        gen = tts.run({
            "text": phrase,
            "text_lang": "ru",
            "ref_audio_path": ref_wav,
            "prompt_text": ref_text,
            "prompt_lang": "ru"
        })
        sr, audio_data = next(gen)
        display(Audio(audio_data, rate=sr))


## Шаг 7. Скачивание готовых весов модели для локального плеера
Упаковываем обученные веса в архив `svetlana_custom_weights.zip` и скачиваем на компьютер.

In [ ]:
import glob
from google.colab import files

!zip -j /content/svetlana_custom_weights.zip \
    /content/GPT-SoVITS/SoVITS_weights/svetlana_plus*.pth \
    /content/GPT-SoVITS/GPT_weights/svetlana_plus*.ckpt

print("💾 Скачивание весов модели...")
files.download('/content/svetlana_custom_weights.zip')
